<img src="https://drive.google.com/thumbnail?id=1kl2OFnF2FADAAKjgZUFPo8dsBQUSIJM7&sz=w1000" alt="Encabezado MLDS" width="100%">

# **Preparación de los Datos**
---

## **0. Integrantes del equipo de trabajo**
---

1. **Kevin Andres Leal Perez** (CC 1000519441)
2. **Dairo Enrique Morales Jimenez** (CC 1006656409)
3. **Sergio Andres Sierra Garcia** (CC 1010026343)

## **1. Carga de datos**
---

En esta fase se realiza la **carga y organización inicial** del dataset RealWaste desde Kaggle, preparando una estructura tabular con la ruta de cada imagen y su etiqueta. Esta estructura facilita el análisis y el procesamiento posterior al centralizar la información en un DataFrame.

- **Fuente de datos**: descarga automatizada con `kagglehub`.
- **Estructura esperada**: carpetas por clase dentro de `RealWaste`.
- **Salida clave**: tabla `df` con columnas `image_path` y `label`, base para la preparación de los datos.

In [ ]:
#!pip install kagglehub

In [ ]:
import os
import keras
import random
import warnings
import kagglehub
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

In [ ]:
from scripts.data_acquisition.main import load_dataset

df, path = load_dataset()
print("Path to dataset files:", path)
print(f"Dataset cargado: {len(df)} imágenes")

## **2. Preparación y Limpieza de los Datos**
---

De manera ilustrativa, se realizará un **rebalanceo del conjunto de datos** de imágenes de residuos mediante sobremuestreo (*oversampling*) de las categorías minoritarias, es decir, aquellas que tienen menos de 700 imágenes.

### **2.1. Distribución de clases**

In [ ]:
variable_objetivo = 'label'

# Distribución de etiquetas
conteos = df.groupby(variable_objetivo).agg({'image_path': 'size'}).sort_values(by='image_path', ascending=False)

plt.bar(conteos.index, conteos['image_path'], color='skyblue')
plt.xlabel('Categorías')
plt.ylabel('Número de imágenes')
plt.title('Distribución de etiquetas en el conjunto de datos (original)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

conteos.style.background_gradient(cmap='Blues').set_caption("Distribución de etiquetas en el conjunto de datos")

### **2.2. Estrategia de rebalanceo**

Para evitar sobreajuste y lograr un mejor balance de las clases, el remuestreo **no se aplicará** a las dos clases más frecuentes (Plastic y Metal). Para todas las demás clases se busca alcanzar alrededor de **700 imágenes**, generando aproximadamente 1 859 imágenes nuevas.

La distribución de probabilidad de generación se ajusta entre las 7 clases restantes, asignando mayor probabilidad a las clases con menores conteos.

In [ ]:
# Sobremuestreo hasta 700 imágenes por categoría
conteos_under700 = conteos[conteos['image_path'] < 700]
conteos_over700 = conteos[conteos['image_path'] >= 700]
n_aumentos = 700 * len(conteos_under700) + conteos_over700['image_path'].sum() - len(df)
print(f"Número de aumentos necesarios: {n_aumentos}")

# Distribución de probabilidades de cada categoría
diferencias = 700 - conteos['image_path']
diferencias = diferencias.clip(lower=0)
probabilidades = diferencias / diferencias.sum()
print("Probabilidades de aumento por categoría:")
print(probabilidades)

### **2.3. Data Augmentation**

Se aplican transformaciones aleatorias a imágenes de las clases minoritarias para generar nuevas muestras. Cada imagen aumentada recibe **una única transformación aleatoria** de las disponibles:

- Brillo, contraste, saturación, matiz
- Rotación, traslación, zoom
- Flip horizontal/vertical
- Corte (*shear*)
- Ruido gaussiano

In [ ]:
# Posibles funciones de Keras para data augmentation
def apply_random_single_transform(image: tf.Tensor) -> tf.Tensor:
    """
    Escoge UNA transformación aleatoria de la lista y la aplica.
    
    Args:
        image: Tensor de forma (H, W, C), dtype float32, valores en [0, 1].
    Returns:
        Tensor transformado de la misma forma.
    """
    transformations = [
        lambda img: keras.layers.RandomBrightness(factor=0.3)(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomContrast(factor=0.3)(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomSaturation(factor=(0.5, 1.5))(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomHue(factor=0.1)(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomRotation(factor=0.15)(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1)(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomZoom(height_factor=0.2)(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomFlip(mode="horizontal")(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomFlip(mode="vertical")(img[tf.newaxis])[0],
        lambda img: keras.layers.RandomShear(x_factor=0.1, y_factor=0.1)(img[tf.newaxis])[0],
        lambda img: keras.layers.GaussianNoise(stddev=0.05)(img[tf.newaxis], training=True)[0],
    ]

    chosen = random.choice(transformations)
    return chosen(image)

In [ ]:
from tqdm import tqdm
df_augmented = df.copy()
# Aumento de imágenes por categoría según las probabilidades calculadas
for i in tqdm(range(n_aumentos)):
    category = np.random.choice(conteos.index, p=probabilidades)
    category_images = df[df[variable_objetivo] == category]['image_path']
    image_base = tf.convert_to_tensor(np.array(Image.open(np.random.choice(category_images))) / 255.0, dtype=tf.float32)
    augmented_image = apply_random_single_transform(image_base)
    # Visualización de la imagen aumentada versus original
    if i < 5:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(image_base.numpy())
        axes[0].set_title("Original")
        axes[0].axis('off')
        axes[1].imshow(augmented_image.numpy())
        axes[1].set_title("Aumentada")
        axes[1].axis('off')
        plt.suptitle(f"Aumento {i+1} - Categoría: {category}")
        plt.show()
    # Adición de la imagen aumentada al dataframe
    new_image_path = f"augmented_{i}_{os.path.basename(np.random.choice(category_images))}"
    df_augmented = pd.concat([df_augmented, pd.DataFrame({'image_path': [new_image_path], 'label': [category]})], ignore_index=True)

In [ ]:
df_augmented.groupby(variable_objetivo).agg({'image_path': 'size'}).sort_values(by='image_path', ascending=False).style.background_gradient(cmap='Blues').set_caption("Distribución de etiquetas después del aumento")

El **rebalanceo de categorías** se realizó mediante *data augmentation* dirigido a las clases con menos de 700 imágenes, aplicando transformaciones aleatorias para incrementar su representación sin depender de nuevas recolecciones. Esta estrategia reduce el sesgo hacia clases mayoritarias y mejora la capacidad del modelo para aprender patrones más equitativos entre etiquetas.

Como resultado, la distribución final por categoría queda más homogénea en `df_augmented`, lo que fortalece la etapa de entrenamiento y evaluación.

## **3. Conclusiones**
---

En este cuaderno se ejecutó el pipeline de **preparación y limpieza** del dataset RealWaste, abordando el desbalance de clases identificado en la fase de entendimiento de los datos.

- Se identificaron **7 clases minoritarias** con menos de 700 imágenes; las 2 clases mayoritarias (Plastic y Metal) no se modificaron.
- Se generaron **1 859 imágenes artificiales** mediante transformaciones aleatorias aplicadas a imágenes existentes (*data augmentation*).
- El dataset resultante (`df_augmented`) presenta una distribución más homogénea, aproximando cada clase a 700 imágenes.
- Las transformaciones aplicadas (brillo, contraste, rotación, zoom, flip, ruido, entre otras) preservan la semántica de los residuos y aumentan la diversidad visual para mejorar la generalización del modelo.
- El dataset preparado queda listo para ser consumido por el pipeline de entrenamiento en `scripts/training/`.

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Santiago Toledo Cortés](https://sites.google.com/unal.edu.co/santiagotoledo-cortes/)
* **Diseño de imágenes:**
    - [Mario Andres Rodriguez Triana](https://www.linkedin.com/in/mario-andres-rodriguez-triana-394806145/).
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*